In [227]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [228]:
df = pd.read_csv("Cricket.csv", sep=",", encoding="ISO-8859-1", header=0)
df.head()

,Player,Span,Mat,Inns,NO,Runs,HS,Ave,BF,SR,100,50,0
0,SR Tendulkar (INDIA),1989-2012,463,452,41,18426,200*,44.83,21367,86.23,49,96,20
1,KC Sangakkara (Asia/ICC/SL),2000-2015,404,380,41,14234,169,41.98,18048,78.86,25,93,15
2,RT Ponting (AUS/ICC),1995-2012,375,365,39,13704,164,42.03,17046,80.39,30,82,20
3,ST Jayasuriya (Asia/SL),1989-2011,445,433,18,13430,189,32.36,14725,91.20,28,68,34
4,DPMD Jayawardene (Asia/SL),1998-2015,448,418,39,12650,144,33.37,16020,78.96,19,77,28


In [229]:
df.size

1027

In [230]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Player  79 non-null     object 
 1   Span    79 non-null     object 
 2   Mat     79 non-null     int64  
 3   Inns    79 non-null     int64  
 4   NO      79 non-null     int64  
 5   Runs    79 non-null     int64  
 6   HS      79 non-null     object 
 7   Ave     79 non-null     float64
 8   BF      79 non-null     int64  
 9   SR      79 non-null     float64
 10  100     79 non-null     int64  
 11  50      79 non-null     int64  
 12  0       79 non-null     int64  
dtypes: float64(2), int64(8), object(3)
memory usage: 8.2+ KB


In [231]:
relevant_data = df[['Player', 'SR', 'Ave']].copy()
relevant_data.head()

,Player,SR,Ave
0,SR Tendulkar (INDIA),86.23,44.83
1,KC Sangakkara (Asia/ICC/SL),78.86,41.98
2,RT Ponting (AUS/ICC),80.39,42.03
3,ST Jayasuriya (Asia/SL),91.20,32.36
4,DPMD Jayawardene (Asia/SL),78.96,33.37


In [232]:
# Remove any potential leading/trailing whitespace from player names
relevant_data['Player'] = relevant_data['Player'].str.strip()
relevant_data.head()

,Player,SR,Ave
0,SR Tendulkar (INDIA),86.23,44.83
1,KC Sangakkara (Asia/ICC/SL),78.86,41.98
2,RT Ponting (AUS/ICC),80.39,42.03
3,ST Jayasuriya (Asia/SL),91.20,32.36
4,DPMD Jayawardene (Asia/SL),78.96,33.37


In [233]:
# Standardize the 'SR' and 'Ave' columns
scaler = StandardScaler()
relevant_data[['SR', 'Ave']] = scaler.fit_transform(relevant_data[['SR', 'Ave']])

In [234]:
kmeans = KMeans(n_clusters=4)
relevant_data['Cluster'] = kmeans.fit_predict(relevant_data[['SR', 'Ave']])

In [235]:
kohli_cluster = relevant_data.loc[relevant_data['Player'].str.contains('V Kohli', case=False), 'Cluster'].values[0]

In [236]:
kohli_cluster

3

In [237]:
same_cluster_players = relevant_data[relevant_data['Cluster'] == kohli_cluster]['Player']
print(same_cluster_players.tolist())

['SR Tendulkar\xa0(INDIA)', 'MS Dhoni\xa0(Asia/INDIA)', 'AB de Villiers\xa0(Afr/SA)', 'V Kohli\xa0(INDIA)', 'MJ Clarke\xa0(AUS)', 'HM Amla\xa0(SA)', 'IVA Richards\xa0(WI)', 'LRPL Taylor\xa0(NZ)', 'ML Hayden\xa0(AUS/ICC)', 'MJ Guptill\xa0(NZ)', 'MEK Hussey\xa0(AUS)', 'RG Sharma\xa0(INDIA)']


In [238]:
dravid_cluster = relevant_data.loc[relevant_data['Player'].str.contains('R Dravid', case=False), 'Cluster'].values[0]
dravid_cluster

1

In [239]:
gayle_cluster = relevant_data.loc[relevant_data['Player'].str.contains('CH Gayle', case=False), 'Cluster'].values[0]
gayle_cluster

0

In [240]:
guptil_cluster = relevant_data.loc[relevant_data['Player'].str.contains('MJ Guptill', case=False), 'Cluster'].values[0]
guptil_cluster

3

In [241]:
relevant_data[relevant_data.SR == min(relevant_data.SR)]

,Player,SR,Ave,Cluster
73,RS Mahanama (SL),-1.898679,-1.535879,1


In [242]:
high_sr_threshold = relevant_data['SR'].median()
high_ave_threshold = relevant_data['Ave'].median()



def categorize_cluster(row):
    if row['SR'] > high_sr_threshold and row['Ave'] > high_ave_threshold:
        return 'A'
    elif row['SR'] <= high_sr_threshold and row['Ave'] <= high_ave_threshold:
        return 'B'
    elif row['SR'] > high_sr_threshold and row['Ave'] <= high_ave_threshold:
        return 'C'
    elif row['SR'] <= high_sr_threshold and row['Ave'] > high_ave_threshold:
        return 'D'

print(f"SR Threshold: {high_sr_threshold}")
print(f"Avg Threshold: {high_ave_threshold}")

SR Threshold: -0.15871735853359795
Avg Threshold: -0.11107538814223279


In [265]:
# Function to categorize clusters based on means
cluster_summary = relevant_data.groupby('Cluster')[["SR", "Ave"]].mean()
cluster_summary

In [269]:
relevant_data['Category'] = relevant_data.apply(categorize_cluster, axis=1)
relevant_data.head()

,Player,SR,Ave,Cluster,Category
0,SR Tendulkar (INDIA),0.703152,1.072294,3,A
1,KC Sangakkara (Asia/ICC/SL),-0.044139,0.587725,1,A
2,RT Ponting (AUS/ICC),0.110997,0.596226,1,A
3,ST Jayasuriya (Asia/SL),1.207091,-1.047909,0,C
4,DPMD Jayawardene (Asia/SL),-0.034000,-0.876185,1,C


In [185]:
relevant_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Player   79 non-null     object 
 1   SR       79 non-null     float64
 2   Ave      79 non-null     float64
 3   Cluster  79 non-null     int32  
dtypes: float64(2), int32(1), object(1)
memory usage: 2.3+ KB


In [285]:
sachin = relevant_data.loc[relevant_data['Player'].str.contains('SR Tendulkar', case=False), 'Category'].values[0]
sachin

'A'

In [287]:
dravid = relevant_data.loc[relevant_data['Player'].str.contains('R Dravid', case=False), 'Category'].values[0]
dravid

'D'

In [289]:
gayle = relevant_data.loc[relevant_data['Player'].str.contains('CH Gayle', case=False), 'Category'].values[0]
gayle

'C'

In [291]:
guptil = relevant_data.loc[relevant_data['Player'].str.contains('MJ Guptill', case=False), 'Category'].values[0]
guptil

'A'

In [271]:
relevant_data[relevant_data.Player.str.contains("SR Tendulkar")]

,Player,SR,Ave,Cluster,Category
0,SR Tendulkar (INDIA),0.703152,1.072294,3,A


In [283]:
relevant_data[relevant_data.Player.str.contains("IVA Richards")]

,Player,SR,Ave,Cluster,Category
42,IVA Richards (WI),1.105695,1.441247,3,A


In [273]:
relevant_data[relevant_data.Player.str.contains("R Dravid")]

,Player,SR,Ave,Cluster,Category
8,R Dravid (Asia/ICC/INDIA),-0.81678,0.108256,1,D


In [275]:
relevant_data[relevant_data.Player.str.contains("CH Gayle")]

,Player,SR,Ave,Cluster,Category
17,CH Gayle (ICC/WI),0.589588,-0.202889,0,C


In [277]:
relevant_data[relevant_data.Player.str.contains("MJ Guptill")]

,Player,SR,Ave,Cluster,Category
63,MJ Guptill (NZ),0.855246,0.812157,3,A
